# 10 — Agentic Systems: Autonomous AI Workflows

## Part 1: Theory

### 1.1 What Are AI Agents?

An agent is an AI system that can **perceive, reason, and act** autonomously to achieve goals.

```
Traditional LLM:  Input → Output (single turn)
Agent:            Goal → [Observe → Think → Act → Observe → Think → Act → ...] → Result
```

### 1.2 Agent Architectures

| Architecture | How It Works | Strengths | Weaknesses |
|-------------|-------------|-----------|------------|
| **ReAct** | Interleave reasoning + action | Simple, interpretable | Can loop |
| **Plan-and-Execute** | Plan all steps first, then execute | Efficient | Brittle plans |
| **LATS** | Tree search over action sequences | Thorough | Expensive |
| **Reflexion** | Execute → reflect on failure → retry | Self-improving | Slow |

### 1.3 Tool Use

Agents extend LLM capabilities by calling **tools** (functions):
```
LLM sees: "search_database(query: str) → Returns relevant documents"
LLM decides: "I need to search for customer info" → calls search_database("customer C-001")
LLM receives: result → incorporates into reasoning
```

### 1.4 Memory Systems

| Type | Duration | Implementation | Use |
|------|----------|---------------|-----|
| **Short-term** | Current conversation | Message history | Context |
| **Long-term** | Persistent | Vector store | Knowledge |
| **Episodic** | Past experiences | Structured logs | Learning from mistakes |
| **Working** | Current task | Scratchpad | Intermediate results |

### 1.5 Multi-Agent Patterns

```
Supervisor:     Boss agent delegates to specialist agents
Debate:         Multiple agents argue, reach consensus
Pipeline:       Agent A → Agent B → Agent C (sequential)
Swarm:          Agents self-organize around tasks
```

### 1.6 LangGraph Concepts

```
State Machine:
  Nodes = Functions that transform state
  Edges = Transitions (conditional or fixed)
  State = Shared data (messages, variables)
  Checkpoints = Saved state for resume/human-in-the-loop
```

### 1.7 Failure Modes & Guardrails

| Failure | Cause | Mitigation |
|---------|-------|------------|
| Infinite loop | Agent keeps retrying same action | Step limit |
| Hallucinated tools | Agent calls non-existent function | Strict tool schema |
| Goal drift | Agent pursues unintended objectives | Output validation |
| Cost explosion | Too many LLM calls | Cost cap |
| Unsafe actions | Agent deletes data, sends emails | Action blocklist + approval |

### 1.8 Production Agent Architecture
```
┌─────────────────────────────────────────────────────────────┐
│                    Agent Orchestrator                        │
├─────────────────────────────────────────────────────────────┤
│  ┌────────┐   ┌─────────┐   ┌──────────┐   ┌───────────┐  │
│  │ Router │──▶│ Planner │──▶│ Executor │──▶│ Validator │  │
│  └────────┘   └─────────┘   └────┬─────┘   └───────────┘  │
│                                   │                         │
│  ┌────────────────────────────────▼──────────────────────┐  │
│  │              Tool Registry                            │  │
│  │  [Search] [Database] [API] [Email] [Calculator]       │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │  Guardrails: Steps≤10 | Cost≤$1 | Blocklist | Timeout │  │
│  └───────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
```

---

## Part 2: Implementation

In [ ]:
import os
import json
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct:free",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0,
)
print("Setup complete")

### Define Tools

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search internal knowledge base for relevant information."""
    kb = {
        "refund": "Refund policy: Full refund within 30 days. Pro-rated after. Takes 5-7 business days.",
        "pricing": "Enterprise: $99/user/month. Includes SSO, audit logs, priority support.",
        "api": "Rate limits: 1000 req/min Enterprise, 100 req/min Free. Auth via Bearer token.",
        "sla": "Enterprise SLA: 99.9% uptime, 4-hour response for critical issues.",
    }
    for key, val in kb.items():
        if key in query.lower():
            return val
    return "No relevant documents found."

@tool
def get_customer(customer_id: str) -> str:
    """Look up customer account information."""
    customers = {
        "C-001": {"name": "Acme Corp", "plan": "Enterprise", "seats": 150, "health": "green"},
        "C-002": {"name": "StartupXYZ", "plan": "Free", "seats": 5, "health": "yellow"},
        "C-003": {"name": "BigBank", "plan": "Enterprise", "seats": 500, "health": "red"},
    }
    return json.dumps(customers.get(customer_id, {"error": "not found"}))

@tool
def create_ticket(subject: str, priority: str, customer_id: str) -> str:
    """Create a support ticket."""
    return json.dumps({"ticket_id": f"TKT-{datetime.now().strftime('%H%M%S')}", "subject": subject, "priority": priority, "status": "created"})

@tool
def send_notification(channel: str, message: str) -> str:
    """Send notification via email or slack."""
    return f"Sent via {channel}: {message[:80]}"

tools = [search_knowledge_base, get_customer, create_ticket, send_notification]
print(f"Tools: {[t.name for t in tools]}")

### ReAct Agent

In [ ]:
from langgraph.prebuilt import create_react_agent

system = """You are an enterprise support agent. For each request:
1. Look up customer info
2. Search knowledge base for relevant policies
3. Take action (create ticket, notify) if needed
4. Provide clear resolution

Always check customer info before acting."""

agent = create_react_agent(llm, tools, state_modifier=system)

result = agent.invoke({"messages": [HumanMessage(content="Customer C-003 says the API has been down for 2 hours. They're on Enterprise plan.")]})

for msg in result["messages"]:
    if hasattr(msg, "content") and msg.content:
        print(f"[{msg.__class__.__name__}] {msg.content[:200]}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[Tool] {tc['name']}({tc['args']})")
    print()

### Multi-Agent Routing

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

def router(state: MessagesState) -> str:
    msg = state["messages"][-1].content.lower()
    if any(w in msg for w in ["billing", "charge", "refund", "invoice"]): return "billing"
    if any(w in msg for w in ["api", "error", "bug", "down", "500"]): return "technical"
    return "general"

def billing_agent(state: MessagesState):
    r = llm.invoke([SystemMessage(content="You are a billing specialist. Be empathetic."), *state["messages"]])
    return {"messages": [r]}

def technical_agent(state: MessagesState):
    r = llm.invoke([SystemMessage(content="You are a technical support engineer. Be precise."), *state["messages"]])
    return {"messages": [r]}

def general_agent(state: MessagesState):
    r = llm.invoke([SystemMessage(content="You are a helpful support agent."), *state["messages"]])
    return {"messages": [r]}

wf = StateGraph(MessagesState)
wf.add_node("billing", billing_agent)
wf.add_node("technical", technical_agent)
wf.add_node("general", general_agent)
wf.add_conditional_edges(START, router)
wf.add_edge("billing", END)
wf.add_edge("technical", END)
wf.add_edge("general", END)
app = wf.compile()

for q in ["I was charged twice!", "API returning 500 errors", "What features do you offer?"]:
    result = app.invoke({"messages": [HumanMessage(content=q)]})
    print(f"Q: {q}\nA: {result['messages'][-1].content[:100]}...\n")

### Agent Guardrails

In [ ]:
class AgentGuardrails:
    def __init__(self, max_steps=10, max_cost=1.0, blocked_actions=None):
        self.max_steps = max_steps
        self.max_cost = max_cost
        self.blocked = blocked_actions or ["delete", "drop", "admin"]
        self.steps = 0
        self.cost = 0.0
        self.log = []
    
    def check(self, action, est_cost=0.01):
        self.steps += 1
        self.cost += est_cost
        self.log.append({"step": self.steps, "action": action, "time": datetime.now().isoformat()})
        
        if self.steps > self.max_steps:
            return {"allowed": False, "reason": f"Step limit ({self.max_steps}) exceeded"}
        if self.cost > self.max_cost:
            return {"allowed": False, "reason": f"Cost cap (${self.max_cost}) exceeded"}
        if any(b in action.lower() for b in self.blocked):
            return {"allowed": False, "reason": f"Blocked action: {action}"}
        return {"allowed": True}
    
    def summary(self):
        return {"steps": self.steps, "cost": f"${self.cost:.4f}", "actions": len(self.log)}

# Demo
guard = AgentGuardrails(max_steps=5, max_cost=0.05)
for action in ["search_kb", "get_customer", "create_ticket", "send_email", "delete_account", "extra_step"]:
    r = guard.check(action)
    print(f"{'✅' if r['allowed'] else '❌'} {action} → {r}")

print(f"\nSummary: {guard.summary()}")

## Part 3: Key Takeaways

1. **Agents = LLM + Tools + Loop** — they reason about WHEN and HOW to use tools
2. **ReAct** is the simplest effective pattern (reason → act → observe → repeat)
3. **Multi-agent routing** directs queries to specialists (cheaper, more accurate)
4. **Guardrails are mandatory** — step limits, cost caps, action blocklists
5. **Human-in-the-loop** for high-stakes actions (refunds, deletions, escalations)
6. **Observe everything** — log every tool call, every decision, every cost

### Production Checklist
- ✅ Step limit (prevent infinite loops)
- ✅ Cost cap (prevent runaway spending)
- ✅ Action blocklist (prevent dangerous operations)
- ✅ Timeout (kill stuck agents)
- ✅ Human approval for high-risk actions
- ✅ Full audit trail of all decisions
- ✅ Fallback to human when agent is stuck

---

## 🎉 Project Complete!

You've built a complete Enterprise AI System covering:
1. ✅ LLM Evaluation & Benchmarking
2. ✅ Model Selection Frameworks
3. ✅ Vector Databases & Embeddings
4. ✅ RAG Pipelines (Naive → Advanced)
5. ✅ MLOps & Drift Detection
6. ✅ Cloud Deployment (Serverless)
7. ✅ Observability & SLOs
8. ✅ Compliance & Privacy
9. ✅ Prompt Engineering
10. ✅ Agentic Systems